# 모델 선택 기준

정확도, 속도, 메모리, 라이선스 조건이 다를 때 어떻게 모델 후보를 비교할까요?

Python 3.12와 잠긴 패키지를 사용합니다. 첫 실행에서는 로컬 모델을 다운로드합니다.

## 비교표 만들기

**입력**은 모델 ID, 대표 작업, 라이선스를 담은 사전 세 개입니다. `task`는 실습에서 사용할 입력·출력 목적이고 `license`는 모델 카드에 표시된 사용 조건입니다. 이 셀은 후보 정보를 한 줄씩 출력할 뿐 성능을 비교하지 않습니다.

In [1]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")
from transformers import AutoModelForCausalLM, AutoTokenizer

candidates = [
    {"id": "Qwen/Qwen2.5-0.5B-Instruct", "task": "text-generation", "license": "apache-2.0"},
    {"id": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", "task": "sentence-similarity", "license": "apache-2.0"},
    {"id": "klue/bert-base", "task": "fill-mask", "license": "CC-BY-SA-4.0"},
]
for item in candidates:
    print(item)

{'id': 'Qwen/Qwen2.5-0.5B-Instruct', 'task': 'text-generation', 'license': 'apache-2.0'}
{'id': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', 'task': 'sentence-similarity', 'license': 'apache-2.0'}
{'id': 'klue/bert-base', 'task': 'fill-mask', 'license': 'CC-BY-SA-4.0'}


## 현재 환경에서 실행 가능성 확인

`AutoTokenizer.from_pretrained()`는 모델 ID에 맞는 tokenizer 객체를 반환하고, `AutoModelForCausalLM.from_pretrained()`는 다음 토큰 생성용 모델 객체를 반환합니다. `num_parameters()`는 파라미터 수를 정수로 반환합니다. 출력의 로딩 시간과 토큰화 시간은 현재 캐시와 CPU에서 측정한 값이며 생성 답변의 지연시간이 아닙니다.

In [2]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
import time
model_id = candidates[0]["id"]
revision = "7ae557604adf67be50417f59c2c2f167def9a775"
tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)
start = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(model_id, revision=revision)
load_seconds = time.perf_counter() - start
text = "한국어 문장을 토큰화합니다. " * 100
start = time.perf_counter()
ids = tokenizer.encode(text)
tokenize_ms = (time.perf_counter() - start) * 1000
print("parameters:", f"{model.num_parameters():,}")
print("cached load seconds:", round(load_seconds, 3))
print("tokenize:", len(ids), "tokens in", round(tokenize_ms, 3), "ms")

parameters: 494,032,768
cached load seconds: 0.384
tokenize: 1202 tokens in 3.631 ms


## 필수 조건으로 후보 거르기

생성 작업과 Apache-2.0을 필수 조건으로 입력합니다. 빈 목록 `eligible`을 만든 뒤 `for` 반복문이 후보 사전을 하나씩 꺼냅니다. 작업 일치 여부와 라이선스 일치 여부를 각각 불리언 값으로 계산하고, 두 값이 모두 `True`일 때만 `append()`로 후보를 목록 끝에 추가합니다. 반복이 끝나면 `eligible`에는 두 조건을 모두 통과한 사전만 남습니다. 후보 하나가 남았다는 사실은 그 후보가 가장 정확하다는 뜻이 아닙니다.

In [3]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
required_task = "text-generation"
required_license = "apache-2.0"
eligible = []
for item in candidates:
    task_matches = item["task"] == required_task
    license_matches = item["license"] == required_license
    if task_matches and license_matches:
        eligible.append(item)
print("eligible:", eligible)
print("선택 전에는 실제 대표 질문과 정답 기준으로 품질과 속도를 추가 측정해야 합니다.")

eligible: [{'id': 'Qwen/Qwen2.5-0.5B-Instruct', 'task': 'text-generation', 'license': 'apache-2.0'}]
선택 전에는 실제 대표 질문과 정답 기준으로 품질과 속도를 추가 측정해야 합니다.


## 구조 검증

예상한 실험 조건과 출력 구조가 유지되었는지 확인합니다.

In [4]:
assert len(candidates) == 3 and len(eligible) == 1
assert model.num_parameters() == 494_032_768
print("검증: 필수 조건으로 3개 후보 중 1개를 걸렀습니다.")

검증: 필수 조건으로 3개 후보 중 1개를 걸렀습니다.
